In [0]:
%pip install faker

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from faker import Faker
import random
from datetime import datetime, timedelta
from pyspark.sql.functions import col, expr

In [0]:
fake = Faker()

In [0]:
max_id_df = spark.read.jdbc(
    jdbc_url,
    "(SELECT MAX(Transaction_ID) AS max_id FROM dbo.FACT_TRANSACTION) t",
    properties=connection_props
)

START_ID = max_id_df.collect()[0]["max_id"]
print("Current max Transaction_ID:", START_ID)


In [0]:
%run /Workspace/Users/aradhana.dubey@diggibyte.com/draft/dim_full_load/Configuration

In [0]:
max_id_df = spark.read.jdbc(
    url=jdbc_url,
    table="(select * from dbo.FACT_TRANSACTION where updated_at >= DATEADD(day, -30, CAST(GETDATE() AS DATE))) as t",
    properties=jdbc_properties
)
display(max_id_df)

In [0]:
#NEW_RECORDS = 499000   # change this as needed


In [0]:
NUM_CUSTOMERS = 100
NUM_BRANCHES = 10
NUM_TRANSACTIONS = 499000

In [0]:
customers_data = []
for i in range(1, NUM_CUSTOMERS + 1):
    customers_data.append((
        i,                          # Customer_ID
        fake.name(),                # Customer_Name
        fake.address().replace('\n', ', '), # Address
        fake.phone_number()         # Contact_Info
    ))

schema_customer = StructType([
    StructField("Customer_ID", IntegerType(), False),
    StructField("Customer_Name", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("Contact_Info", StringType(), True)
])

df_customer = spark.createDataFrame(customers_data, schema_customer)

# Write to SQL Server
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"
connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
df_customer.write.jdbc(url=jdbc_url, table="dbo.DIM_CUSTOMER", mode="overwrite", properties=connection_props)

In [0]:
# Use the same jdbc_url and connection_props you defined earlier
df_from_sql = spark.read.jdbc(
    url=jdbc_url, 
    table="dbo.DIM_CUSTOMER", 
    properties=connection_props
)

# Display the result in a table format
display(df_from_sql)

In [0]:
branches_data = []
regions = ['North', 'South', 'East', 'West']
for i in range(1, NUM_BRANCHES + 1):
    branches_data.append((
        i,                          # Branch_ID
        f"{fake.city()} Branch",    # Branch_Name
        fake.city(),                # Location
        random.choice(regions)      # Region
    ))

schema_branch = StructType([
    StructField("Branch_ID", IntegerType(), False),
    StructField("Branch_Name", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("Region", StringType(), True)
])

df_branch = spark.createDataFrame(branches_data, schema_branch)

# Write to SQL Server 
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"
connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
df_branch.write.jdbc(url=jdbc_url, table="dbo.DIM_BRANCH", mode="overwrite", properties=connection_props)

In [0]:
print("Generating DIM_TRANSACTION_TYPE...")
types_data = [
    (1, "Deposit"),
    (2, "Withdrawal"),
    (3, "Transfer"),
    (4, "Payment")
]

schema_type = StructType([
    StructField("Type_ID", IntegerType(), False),
    StructField("Transaction_Type", StringType(), True)
])

df_type = spark.createDataFrame(types_data, schema_type)

# Write to SQL Server 
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"
connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
df_type.write.jdbc(url=jdbc_url, table="dbo.DIM_TRANSACTION_TYPE", mode="overwrite", properties=connection_props)

In [0]:
status_data = [
    (1, "Completed", "Transaction successful"),
    (2, "Pending", "Awaiting bank clearance"),
    (3, "Failed", "Insufficient funds or error"),
    (4, "Cancelled", "User cancelled transaction")
]

schema_status = StructType([
    StructField("Status_ID", IntegerType(), False),
    StructField("Status_Description", StringType(), True), # Mapped from image text
    StructField("Detailed_Description", StringType(), True)
])

df_status = spark.createDataFrame(status_data, schema_status)

jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"
connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df_status.write.jdbc(url=jdbc_url, table="dbo.DIM_TRANSACTION_STATUS", mode="overwrite", properties=connection_props)

In [0]:
# Generate dates for the current year
start_date = datetime(2025, 1, 1)
date_data = []
for i in range(365):
    current_date = start_date + timedelta(days=i)
    date_id = int(current_date.strftime('%Y%m%d')) # Format: 20230101
    
    date_data.append((
        date_id,                        # Date_ID
        current_date.date(),            # Transaction_Date (Date object)
        current_date.day,               # Day
        current_date.strftime('%B'),    # Month
        current_date.year               # Year
    ))

schema_date = StructType([
    StructField("Date_ID", IntegerType(), False),
    StructField("Transaction_Date", DateType(), True),
    StructField("Day", IntegerType(), True),
    StructField("Month", StringType(), True),
    StructField("Year", IntegerType(), True)
])

df_date = spark.createDataFrame(date_data, schema_date)

# Write to SQL Server 
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"
connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
df_date.write.jdbc(url=jdbc_url, table="dbo.DIM_DATE", mode="overwrite", properties=connection_props)

In [0]:
# GET DIM COUNTS (SAFE)
customer_count = df_customer.count()
branch_count = df_branch.count()
type_count = df_type.count()
status_count = df_status.count()

# GENERATE FACT DATA USING SPARK
df_fact = (
    spark.range(1, NUM_TRANSACTIONS + 1)
    .withColumn("Transaction_ID", col("id").cast("int"))
    .withColumn(
        "Account_Number",
        expr("lpad(cast(cast(rand()*1e15 as bigint) as string), 15, '0')")
    )
    .withColumn(
        "Customer_ID",
        expr(f"cast(rand()*{customer_count} as int) + 1")
    )
    .withColumn(
        "Type_ID",
        expr(f"cast(rand()*{type_count} as int) + 1")
    )
    .withColumn(
        "Amount",
        expr("round(rand()*4990 + 10, 2)")
    )
    .withColumn(
        "Date_ID",
        expr("20250101 + cast(rand()*364 as int)")
    )
    .withColumn(
        "Balance_After",
        expr("round(rand()*50000, 2)")
    )
    .withColumn(
        "Branch_ID",
        expr(f"cast(rand()*{branch_count} as int) + 1")
    )
    .withColumn(
        "Status_ID",
        expr(f"cast(rand()*{status_count} as int) + 1")
    )
    .drop("id")
)

df_fact.printSchema()
df_fact.show(5)

df_fact.write.jdbc(
    url=jdbc_url,
    table="dbo.FACT_TRANSACTION",
    mode="overwrite",
    properties=connection_props
)


In [0]:
from pyspark.sql.functions import col, expr, current_timestamp, date_format, lit

df_fact_incremental = (
    spark.range(1, NUM_TRANSACTIONS + 1)
    .withColumn("Transaction_ID", (col("id") + START_ID).cast("int"))
    .withColumn(
        "Account_Number",
        expr("lpad(cast(cast(rand()*1e15 as bigint) as string), 15, '0')")
    )
    .withColumn(
        "Customer_ID",
        expr(f"cast(rand()*{customer_count} as int) + 1")
    )
    .withColumn(
        "Type_ID",
        expr(f"cast(rand()*{type_count} as int) + 1")
    )
    .withColumn(
        "Amount",
        expr("round(rand()*4990 + 10, 2)")
    )
    .withColumn(
        "Date_ID",
        expr("20250101 + cast(rand()*364 as int)")
    )
    .withColumn(
        "Balance_After",
        expr("round(rand()*50000, 2)")
    )
    .withColumn(
        "Branch_ID",
        expr(f"cast(rand()*{branch_count} as int) + 1")
    )
    .withColumn(
        "Status_ID",
        expr(f"cast(rand()*{status_count} as int) + 1")
    )

    # 🔑 SQL Server–friendly columns
    .withColumn(
        "created_at",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "updated_at",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "is_deleted",
        lit(0).cast("int")   # bit compatible
    )

    .drop("id")
)


In [0]:
from pyspark.sql.functions import col

df_fact_incremental_fixed = (
    df_fact_incremental
    .withColumn("created_at", col("created_at").cast("string"))   # datetime2 safe
    .withColumn("updated_at", col("updated_at").cast("string"))   # datetime2 safe
    .withColumn("is_deleted", col("is_deleted").cast("int"))      # bit safe
)


In [0]:
df_fact.write.jdbc(
    url=jdbc_url,
    table="dbo.FACT_TRANSACTION",
    mode="overwrite",   # ✅ ONLY for initial/reset load
    properties=connection_props
)
display(df_fact)
